# Data Preprocessing System

Notebook untuk membaca, memvalidasi, dan membersihkan kolom komentar dari file CSV atau Excel.

**Alur:** Input Data → Validasi → Pembersihan Teks → Output DataFrame

## Setup

Jalankan sekali jika paket belum terpasang:

```bash
pip install -r requirements.txt
```

In [4]:
import os
import re
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz, process
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 50)

## Konfigurasi

Ubah path file dan nama kolom komentar di sini. Untuk Excel, gunakan ekstensi `.xlsx`.

In [5]:
FILE_PATH = "sample-data.csv"  # atau "data.xlsx"
SHEET_NAME = 0  # indeks/nama sheet untuk Excel
FILE_ENCODING = None  # None = deteksi otomatis; atau "utf-8", "cp1252", dll.

# Pilih kolom komentar: indeks (0-based), nama kolom, atau None untuk widget
COMMENT_COLUMNS = [1]  # contoh: [1] = kolom kritik/saran; [0, 1] = gabung keduanya
MERGED_COLUMN_NAME = "merged_comment"

---
## 1. Input Data

Membaca file, mendeteksi kolom, menampilkan nama kolom, dan menggabungkan kolom komentar yang dipilih.

In [6]:
def read_csv_with_encoding(path: Path, encoding: str | None = None) -> tuple[pd.DataFrame, str]:
    """Baca CSV dengan encoding eksplisit atau fallback otomatis."""
    if encoding:
        return pd.read_csv(path, encoding=encoding), encoding

    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Tidak dapat membaca {path} dengan encoding yang didukung.")


def read_data_file(
    file_path: str,
    sheet_name=0,
    encoding: str | None = None,
) -> pd.DataFrame:
    """Membaca CSV atau Excel (.xlsx) ke DataFrame."""
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df, used_encoding = read_csv_with_encoding(path, encoding=encoding)
        print(f"CSV dibaca dengan encoding: {used_encoding}")
        return df
    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
    raise ValueError(f"Format tidak didukung: {suffix}. Gunakan .csv atau .xlsx")


def detect_columns(df: pd.DataFrame) -> list[str]:
    """Mengembalikan daftar semua nama kolom."""
    return df.columns.tolist()


def display_column_info(df: pd.DataFrame) -> None:
    """Menampilkan informasi kolom untuk membantu pemilihan."""
    print(f"Jumlah baris: {len(df)}")
    print(f"Jumlah kolom: {len(df.columns)}\n")
    print("Daftar kolom:")
    for i, col in enumerate(df.columns, start=1):
        non_null = df[col].notna().sum()
        print(f"  {i}. {col!r}  (non-null: {non_null})")


def merge_comment_columns(
    df: pd.DataFrame,
    comment_columns: list[str],
    merged_name: str = "merged_comment",
    separator: str = " ",
) -> pd.DataFrame:
    """Menggabungkan beberapa kolom komentar menjadi satu kolom teks."""
    missing = [c for c in comment_columns if c not in df.columns]
    if missing:
        raise KeyError(f"Kolom tidak ditemukan: {missing}")

    result = df.copy()
    parts = [result[col].astype(str).str.strip() for col in comment_columns]
    result[merged_name] = parts[0]
    for part in parts[1:]:
        result[merged_name] = result[merged_name].str.cat(part, sep=separator)
    return result

In [7]:
# --- Baca file & tampilkan kolom ---
raw_df = read_data_file(FILE_PATH, sheet_name=SHEET_NAME, encoding=FILE_ENCODING)
all_columns = detect_columns(raw_df)
display_column_info(raw_df)

print("\nPreview data mentah:")
display(raw_df.head())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x94 in position 4226: invalid start byte

In [ ]:
def select_comment_columns_interactive(columns: list[str]):
    """Widget interaktif untuk memilih satu atau lebih kolom komentar."""
    if not HAS_WIDGETS:
        raise RuntimeError(
            "ipywidgets tidak tersedia. Set COMMENT_COLUMNS di cell Konfigurasi."
        )

    options = [(f"[{i}] {col}", col) for i, col in enumerate(columns)]
    selector = widgets.SelectMultiple(
        options=options,
        description="Kolom:",
        rows=min(len(columns), 8),
        layout=widgets.Layout(width="100%"),
    )
    display(
        widgets.VBox([
            widgets.HTML("<b>Pilih kolom komentar (Ctrl+klik untuk multi-select):</b>"),
            selector,
        ])
    )
    return selector


if COMMENT_COLUMNS is None and HAS_WIDGETS:
    print("Mode interaktif — pilih kolom di widget, lalu jalankan ulang cell ini.")
    _widget = select_comment_columns_interactive(all_columns)
    selected_cols = list(_widget.value) if _widget.value else list(all_columns)
elif COMMENT_COLUMNS is None:
    selected_cols = list(all_columns)
    print("COMMENT_COLUMNS = None & widget tidak tersedia -> semua kolom dipakai.")
else:
    selected_cols = resolve_comment_columns(all_columns, COMMENT_COLUMNS)

print("Kolom komentar yang akan dianalisis:", selected_cols)

input_df = merge_comment_columns(raw_df, selected_cols, merged_name=MERGED_COLUMN_NAME)
print(f"\nKolom gabungan '{MERGED_COLUMN_NAME}' dibuat dari {len(selected_cols)} kolom.")
display(input_df[[MERGED_COLUMN_NAME]].head(10))

---
## 2. Validasi Data

Memeriksa keberadaan file, file kosong, dan membersihkan nilai yang hilang/tidak relevan.

In [ ]:
EMPTY_MARKERS = {"-", "—", "–", "tidak ada", "tdk ada", "ga ada", "gak ada", "none", "n/a", "na", ""}


def validate_file_exists(file_path: str) -> Path:
    """Memastikan file ada; raise FileNotFoundError jika tidak."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path.resolve()}")
    if path.stat().st_size == 0:
        raise ValueError(f"File kosong (0 byte): {path.resolve()}")
    return path


def validate_dataframe(df: pd.DataFrame, text_column: str) -> pd.DataFrame:
    """Validasi isi DataFrame dan bersihkan baris/karakter tidak valid."""
    if df.empty:
        raise ValueError("DataFrame kosong — tidak ada baris data.")
    if text_column not in df.columns:
        raise KeyError(f"Kolom teks '{text_column}' tidak ada.")

    cleaned = df.copy()
    cleaned[text_column] = cleaned[text_column].astype(str).str.strip()

    # Normalisasi placeholder kosong ke NaN
    cleaned[text_column] = cleaned[text_column].replace({"nan": pd.NA, "None": pd.NA})
    mask_empty = cleaned[text_column].str.lower().isin(EMPTY_MARKERS)
    cleaned.loc[mask_empty, text_column] = pd.NA

    before = len(cleaned)
    cleaned = cleaned.dropna(subset=[text_column])
    after = len(cleaned)

    print(f"Validasi file: OK")
    print(f"Baris sebelum pembersihan: {before}")
    print(f"Baris setelah hapus komentar kosong/placeholder: {after}")
    print(f"Baris dihapus: {before - after}")

    if cleaned.empty:
        raise ValueError("Semua baris komentar kosong setelah validasi.")

    return cleaned.reset_index(drop=True)

In [ ]:
validate_file_exists(FILE_PATH)
validated_df = validate_dataframe(input_df, MERGED_COLUMN_NAME)

print("\nData setelah validasi:")
display(validated_df[[MERGED_COLUMN_NAME]].head(10))

---
## 3. Pembersihan Teks

Pipeline: lowercase → hapus emoticon → hapus angka → hapus tanda baca → rapikan spasi →
reduksi karakter berulang → normalisasi slang → koreksi typo → tokenisasi → stemming (Sastrawi).

In [ ]:
# Kamus slang Indonesia umum → bentuk baku
SLANG_MAP = {
    "gk": "tidak", "gak": "tidak", "ga": "tidak", "nggak": "tidak", "ngga": "tidak",
    "tdk": "tidak", "g": "tidak",
    "udh": "sudah", "udah": "sudah", "dh": "sudah",
    "blm": "belum", "belom": "belum",
    "yg": "yang", "dgn": "dengan", "krn": "karena", "krna": "karena",
    "bgt": "banget", "bngt": "banget", "bgtt": "banget",
    "lebi": "lebih", "lbih": "lebih",
    "bayak": "banyak", "byk": "banyak",
    "tlong": "tolong", "tlng": "tolong",
    "bnyk": "banyak", "dpt": "dapat", "jd": "jadi", "jg": "juga",
    "sy": "saya", "gw": "saya", "gue": "saya", "lu": "kamu",
    "pdhl": "padahal", "klo": "kalau", "kl": "kalau", "kalo": "kalau",
    "trs": "terus", "trus": "terus", "tp": "tapi", "tpi": "tapi",
    "dpt": "dapat", "bs": "bisa", "bisa": "bisa",
    "mantap": "bagus", "keren": "bagus", "kece": "bagus",
    "pede": "percaya diri", "pd": "percaya diri",
    "acara": "acara", "proker": "program kerja",
}

# Kosakata referensi untuk koreksi typo (RapidFuzz)
TYPO_VOCABULARY = sorted(set([
    "tidak", "ada", "sudah", "lebih", "banyak", "panitia", "acara", "program", "kerja",
    "kritik", "saran", "selanjutnya", "bagus", "seru", "meriah", "dingin", "kursi",
    "microphone", "mic", "dokumentasi", "performance", "keberanian", "percaya", "diri",
    "kepemimpinan", "integritas", "totalitas", "disiplin", "semangat", "kerjasama",
    "pantang", "menyerah", "kegiatan", "sekolah", "guru", "siswa", "osis", "mpk",
    "audience", "audiens", "tampilan", "durasi", "band", "debat", "drama", "impiian",
    "impian", "inspirasi", "motivasi", "kegigihan", "leadership", "serviam", "humanis",
    "global", "kedisiplinan", "kecerdasan", "kritis", "optimis", "gotong", "royong",
    "kompak", "supportif", "tanggung", "jawab", "presentasi", "musik", "performer",
    "tertib", "ramai", "break", "duduk", "kendala", "lancar", "cepat", "sempurna",
    "konsisten", "rajin", "jujur", "bakal", "bakat", "talent", "contes", "nyanyi",
    "wild", "card", "unjuk", "takut", "coba", "proses", "mimpi", "prestasi",
    "kesopanan", "disiplin", "peraturan", "inspirasi", "motivasi", "kepemimpinan",
    "keberanian", "kepercayaan", "kecerdasan", "berpikir", "kritis", "peduli", "hormat",
    "dokumentasi", "rekam", "momen", "angle", "rencana", "rencana", "perhatikan",
    "persiapkan", "alat", "permusikan", "menunjukkan", "terbaik", "esok", "darah",
    "mengalir", "bersabar", "kerja", "keras", "meriah", "qn", "maybe", "jangan",
    "malu", "menyerah", "pemimpin", "wawasan", "kesempatan", "selanjutnya", "dipilih",
    "tolong", "banget", "seruu", "pol", "mati", "sering", "tambah", "tambahin",
    "tertibkan", "anak", "agak", "ramai", "dingin", "lagi", "ulangi", "semuanya",
    "sangat", "semoga", "ikut", "kedepannya", "pertahankan", "cukup", "sempurna",
    *SLANG_MAP.values(),
]))

# Mapping typo spesifik yang sering muncul di data survei
TYPO_FIXES = {
    "panit": "panitia", "panitiaa": "panitia",
    "pantabg": "pantang", "pantang": "pantang",
    "impuan": "impian", "impian": "impian",
    "kepimpinan": "kepemimpinan", "kepemimpinann": "kepemimpinan",
    "keberanihan": "keberanian", "keberaniann": "keberanian",
    "kedisiplinan": "kedisiplinan", "disiplinan": "disiplin",
    "audience": "audiens", "audiens": "audiens",
    "kendal": "kendala", "kendala": "kendala",
    "udhan": "sudah", "debat": "debat", "inggris": "inggris",
    "klu": "kalau", "dongg": "dong", "yng": "yang", "kmrin": "kemarin",
    "polll": "pol", "seruu": "seru", "bgt": "banget",
    "gaada": "tidak ada", "tdkada": "tidak ada",
    "setengah2": "setengah setengah", "setengah": "setengah",
    "proker": "program kerja",
}

stemmer = StemmerFactory().create_stemmer()

In [ ]:
EMOTICON_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "\U0000FE00-\U0000FE0F"
    "]+",
    flags=re.UNICODE,
)
ASCII_EMOTICON_PATTERN = re.compile(
    r":-\)|:\(|;-?\)|:D|:P|:p|XD|xD|<3",
    flags=re.IGNORECASE,
)

PUNCTUATION_PATTERN = re.compile(r"[^\w\s]", re.UNICODE)
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{1,}")  # gakkk -> gak


def to_lowercase(text: str) -> str:
    return text.lower()


def remove_emoticons(text: str) -> str:
    return ASCII_EMOTICON_PATTERN.sub(" ", EMOTICON_PATTERN.sub(" ", text))


def remove_irrelevant_numbers(text: str) -> str:
    """Menghapus angka standalone yang tidak relevan untuk analisis teks."""
    return re.sub(r"\b\d+\b", " ", text)


def remove_punctuation(text: str) -> str:
    return PUNCTUATION_PATTERN.sub(" ", text)


def remove_excess_space(text: str) -> str:
    return " ".join(text.split())


def remove_repeating_chars(text: str) -> str:
    """Mengurangi karakter berulang dalam kata: gakkk -> gak, polll -> pol."""
    words = text.split()
    collapsed = [REPEAT_CHAR_PATTERN.sub(r"\1", w) for w in words]
    return " ".join(collapsed)


def normalize_slang(text: str, slang_map: dict = SLANG_MAP) -> str:
    words = text.split()
    return " ".join(slang_map.get(w, w) for w in words)


def correct_typos(
    text: str,
    typo_fixes: dict = TYPO_FIXES,
    vocabulary: list = TYPO_VOCABULARY,
    score_cutoff: int = 82,
) -> str:
    """Koreksi typo dengan kamus eksplisit + fuzzy matching (RapidFuzz)."""
    words = text.split()
    corrected = []
    for word in words:
        if word in typo_fixes:
            corrected.append(typo_fixes[word])
            continue
        if word in vocabulary:
            corrected.append(word)
            continue
        match = process.extractOne(word, vocabulary, scorer=fuzz.ratio, score_cutoff=score_cutoff)
        corrected.append(match[0] if match else word)
    return " ".join(corrected)


def tokenize(text: str) -> list[str]:
    """Tokenisasi berbasis spasi setelah normalisasi."""
    return [t for t in text.split() if t.strip()]


def stem_tokens(tokens: list[str], stemmer_instance=stemmer) -> list[str]:
    return [stemmer_instance.stem(t) for t in tokens]


def clean_text_pipeline(text: str) -> dict:
    """Menjalankan seluruh langkah pembersihan dan mengembalikan hasil tiap tahap."""
    if pd.isna(text) or not str(text).strip():
        return {
            "original": text,
            "cleaned_text": "",
            "tokens": [],
            "stemmed_tokens": [],
            "stemmed_text": "",
        }

    s = str(text)
    s = to_lowercase(s)
    s = remove_emoticons(s)
    s = remove_irrelevant_numbers(s)
    s = remove_punctuation(s)
    s = remove_excess_space(s)
    s = remove_repeating_chars(s)
    s = normalize_slang(s)
    s = correct_typos(s)
    s = remove_excess_space(s)

    tokens = tokenize(s)
    stemmed = stem_tokens(tokens)

    return {
        "original": text,
        "cleaned_text": s,
        "tokens": tokens,
        "stemmed_tokens": stemmed,
        "stemmed_text": " ".join(stemmed),
    }


def preprocess_dataframe(df: pd.DataFrame, text_column: str) -> pd.DataFrame:
    """Terapkan pipeline pembersihan ke seluruh kolom komentar."""
    results = df[text_column].apply(clean_text_pipeline)
    output = df.copy()
    output["original_comment"] = results.apply(lambda r: r["original"])
    output["cleaned_text"] = results.apply(lambda r: r["cleaned_text"])
    output["tokens"] = results.apply(lambda r: r["tokens"])
    output["stemmed_tokens"] = results.apply(lambda r: r["stemmed_tokens"])
    output["stemmed_text"] = results.apply(lambda r: r["stemmed_text"])
    return output

In [ ]:
processed_df = preprocess_dataframe(validated_df, MERGED_COLUMN_NAME)

print("=" * 60)
print("HASIL PREPROCESSING")
print("=" * 60)
print(f"Total baris: {len(processed_df)}")
print(f"Kolom output: {list(processed_df.columns)}\n")

display_cols = ["original_comment", "cleaned_text", "tokens", "stemmed_text"]
display(processed_df[display_cols].head(15))

print("\nContoh perbandingan sebelum vs sesudah (5 baris pertama):")
for i, row in processed_df.head(5).iterrows():
    print(f"\n[{i + 1}] ASLI     : {row['original_comment']}")
    print(f"    BERSIH   : {row['cleaned_text']}")
    print(f"    STEM     : {row['stemmed_text']}")

---
## 4. Simpan Output (Opsional)

Simpan DataFrame hasil ke CSV atau Excel.

In [ ]:
SAVE_OUTPUT = False
OUTPUT_PATH = "processed-data.csv"

if SAVE_OUTPUT:
    export_df = processed_df.copy()
    export_df["tokens"] = export_df["tokens"].apply(lambda x: " ".join(x) if isinstance(x, list) else x)
    export_df["stemmed_tokens"] = export_df["stemmed_tokens"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else x
    )
    if OUTPUT_PATH.endswith(".xlsx"):
        export_df.to_excel(OUTPUT_PATH, index=False, engine="openpyxl")
    else:
        export_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Disimpan ke: {OUTPUT_PATH}")
else:
    print("SAVE_OUTPUT = False — lewati penyimpanan. Set True untuk export.")